# 课后练习解答（05.02_environment_and_model_loading）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** DeepSeek 的 tokenizer 在 transformers 5.x 下 AutoTokenizer 返回空 input_ids，最稳妥的修复是？
A. 用 PreTrainedTokenizerFast(tokenizer_file="tokenizer.json") 直接加载
B. 换用 GPT2Tokenizer
C. 禁用 tokenizer
D. 增大 max_length

**解答：** A

**解析：** 直接加载 tokenizer.json 可绕开与 transformers 版本不兼容的自定义分词器 Python 代码。


### 问题2（单选题）

**题目：** device_map="auto" 与 device_map="npu:0" 的关键区别是？
A. auto 可能自动跨设备切分，npu:0 强制单卡
B. 两者完全相同
C. auto 一定更快
D. npu:0 会自动均衡多卡

**解答：** A

**解析：** auto 根据显存估算自动分配层到多设备；显式 npu:0 把全部层放到指定卡。


### 问题3（多选题）

**题目：** 加载 6.9B 参数模型时，显存消耗主要来自？
A. bf16 权重
B. KV cache
C. 中间激活
D. tokenizer 词表文件

**解答：** ABC

**解析：** tokenizer 词表通常只有几 MB~几十 MB，不是显存主要来源。


### 问题4（多选题）

**题目：** SDPA 相比 Eager Attention 的优势包括？
A. 融合 attention kernel
B. 无需额外安装 flash-attn
C. 显存复杂度可降至 O(N)（FlashAttention 类实现）
D. 所有场景精度完全一致

**解答：** ABC

**解析：** SDPA 是 PyTorch 原生 API，CUDA/NPU 可路由到融合内核；数值行为仍可能有微小差异。


### 问题5（判断题）

**题目：** attn_implementation="sdpa" 在 NPU 上必须预装 flash-attn 才能运行。

**解答：** 错

**解析：** NPU 上由 torch_npu 把 SDPA 路由到昇腾融合注意力算子，无需安装 CUDA 版 flash-attn。


### 问题6（判断题）

**题目：** DeepSeek tokenizer 默认没有可用的 pad_token，加载后应显式设置。

**解答：** 对

**解析：** 不设置 pad_token 会在批量填充或生成时出现缺 token 异常。


### 问题7（填空题）

**题目：** 6,914,297,856 参数以 bf16 保存，权重占用约 ____ GB。

**解答：** 约 13.8~14 GB


### 问题8（填空题）

**题目：** AutoModelForCausalLM.from_pretrained 中控制多卡自动切分的参数是 ____，控制低精度加载的参数是 ____。

**解答：** device_map；torch_dtype


### 问题9（简答题）

**题目：** 为什么加载 DeepSeek 模型常需要 trust_remote_code=True？

**解答：** DeepSeek 仓库包含自定义 modeling/tokenization 代码，transformers 需要执行远程代码才能构建模型；trust_remote_code=True 表示允许执行该代码，使用前应核对来源。


### 问题10（简答题）

**题目：** 如何验证 tokenizer 与 model 已正确加载并真正位于 NPU？

**解答：** 打印 tokenizer 的 pad/eos/bos 是否有效，对样例 prompt 编码后检查 input_ids 非空；打印 model.device 与参数 device，执行一次 NPU 前向并回拷结果，确认无 CPU 隐式回退。


### 问题11（代码设计题）

**题目：** 编写 load_model_and_tokenizer(model_id)，要求 bf16、device_map 可控、打印参数量与 device。

**解答：** ```python
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_and_tokenizer(model_id, device_map="auto"):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map=device_map,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    params = sum(p.numel() for p in model.parameters())
    print("params", params, "device", model.device)
    return model, tokenizer
```


### 问题12（单选题）

**题目：** generate 返回空字符串，优先排查顺序正确的是？
A. pad/eos/bos 设置 → 生成参数 → 输入格式
B. 显存 → 数据集 → 优化器
C. 学习率 → 数据增强
D. 模型结构

**解答：** A

**解析：** 空输出通常由 token 设置或停止条件导致，与数据集和优化器无关。


### 问题13（多选题）

**题目：** 控制生成多样性的参数包括？
A. do_sample
B. temperature
C. top_p
D. max_new_tokens

**解答：** ABC

**解析：** max_new_tokens 控制长度，不控制多样性。


### 问题14（判断题）

**题目：** SDPA 通过 PyTorch 原生 API 在 CUDA 与 NPU 上均可路由到融合注意力实现，是跨平台加速方案。

**解答：** 对

**解析：** CUDA 侧自动选择 FlashAttention/Memory-Efficient，NPU 侧由 torch_npu 路由到融合算子。


### 问题15（简答题）

**题目：** 估算 7B bf16 模型在 max_seq_len=2048、batch=1 时 KV cache 的大致开销，并说明计算方式。

**解答：** 以 28 层、32 头、head_dim=128、bf16、batch=1、seq_len=2048 估算：KV 元素数 = 28×2×32×128×2048 ≈ 4.70 亿，bf16 下约 0.94 GB；若按 32 层计算约 1.07 GB。实际应按层数、头数、head_dim 与 dtype 精确计算。
